# Try the Arabizi → Arabic model

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BackendNeuron/egyptian-arabizi-franco/blob/main/infer_colab.ipynb)

Loads the trained model from your Google Drive (`MyDrive/egyptian-arabizi-franco/runs/best.pt`, made by [train_colab.ipynb](train_colab.ipynb)) and runs a few tests. No GPU needed: it runs on Colab's free CPU runtime.

Runtime → **Run all**. The last cell has a box to type your own Arabizi.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!git clone -q https://github.com/BackendNeuron/egyptian-arabizi-franco.git /content/egyptian-arabizi-franco || (cd /content/egyptian-arabizi-franco && git pull -q)
%cd /content/egyptian-arabizi-franco

In [ ]:
import os, time, torch
from training.model import Transliterator

MODEL = '/content/drive/MyDrive/egyptian-arabizi-franco/runs/best.pt'  # change to try another checkpoint, e.g. a backup like egyptian-arabizi-franco-v1-73pct.pt
if not os.path.exists(MODEL):
    # no model on Drive: upload a .pt file instead
    from google.colab import files
    print('No model at', MODEL, '- choose a .pt file to upload')
    MODEL = next(iter(files.upload()))

ck = torch.load(MODEL, map_location='cpu')
model = Transliterator(**ck['config'])
model.load_state_dict(ck['model'])
model.eval()
print(f"loaded {MODEL} · {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M parameters · trained {ck['epoch'] + 1} epochs")


def convert(texts):
    '''Arabizi -> Arabic: a list in, a list out.'''
    return model.transliterate(list(texts))


def show(rows, want=None):
    out = convert(rows)
    for i, (s, p) in enumerate(zip(rows, out)):
        mark = '' if want is None else ('  ✓' if p == want[i] else f'  ✗ want {want[i]}')
        print(f'{s:<38} →  {p}{mark}')

## Test 1 · everyday phrases
Common chat lines, with the Arabic we expect.

In [ ]:
phrases = {
    'ezayak 3amel eh': 'ازيك عامل ايه',
    'saba7 el5eer': 'صباح الخير',
    'ana raye7 el shoghl bokra': 'انا رايح الشغل بكرة',
    'el gaw 7ar awy elnaharda': 'الجو حر اوي النهاردة',
    'msh 3aref a3mel eh': 'مش عارف اعمل ايه',
    'yalla bina': 'يلا بينا',
    'wa7ashtni awy': 'وحشتني اوي',
    'ana ga3an gedan': 'انا جعان جدا',
    'kol sana w enta tayeb': 'كل سنة وانت طيب',
    'el7amdolellah tamam': 'الحمد لله تمام',
}
show(list(phrases), list(phrases.values()))

## Test 2 · one sentence, many spellings
People type the same words differently. The output should stay the same.

In [ ]:
show(['ezayak 3amel eh', 'ezzayak 3amel eh', 'izayak 3amil eh', 'ezayak aamel eih', 'EZAYAK 3AMEL EH', 'ezayak 3amel ehhh'])

## Test 3 · English words mixed in
Work and tech words should stay in Latin letters: `el meeting` → `الmeeting`.

In [ ]:
show(['3andy el meeting bokra', 'eb3atly el link', 'el password ghalat', 'nazzel el update', 'ana fel office dlwa2ty'])

## Test 4 · words it has probably never seen
Names, rare words and new slang: the hardest case.

In [ ]:
show(['mohamed salah gab goal', 'ra7 el mansoura embare7', 'el mosalsal da 3amel trend', 'el kahraba et2ata3et tany', 'el sandwich kan 7ar nar'])

## Test 5 · score on the held-out test set
The hand-written sentences the model never trained on (the same numbers as the training notebook).

In [ ]:
rows = [r.split('\t') for r in open('data/test.tsv', encoding='utf-8').read().splitlines()[1:]]
pred = convert([r[0] for r in rows])
words = [pw == tw for p, r in zip(pred, rows) for pw, tw in zip(p.split(), r[1].split())]
print(f'whole sentences right: {sum(p == r[1] for p, r in zip(pred, rows)) / len(rows):.1%}')
print(f'words right:           {sum(words) / len(words):.1%}')
print()
print('the ones it missed:')
for p, r in zip(pred, rows):
    if p != r[1]:
        print(f'  {r[0]:<34} →  {p:<28} want {r[1]}')

## Test 6 · speed on a CPU
How fast it would be in the pip package or a small server, with no GPU.

In [ ]:
batch = ['ana raye7 el shoghl bokra'] * 50
convert(batch[:2])  # warm up
t0 = time.time(); convert(batch[:1]); one = time.time() - t0
t0 = time.time(); convert(batch); many = time.time() - t0
print(f'one sentence: {one * 1000:.0f} ms · 50 sentences in a batch: {many * 1000:.0f} ms ({many / 50 * 1000:.1f} ms each)')

## Try your own
Type Arabizi in the box and run the cell. For several lines, separate them with `|`.

In [ ]:
text = "ana msh fahem 7aga | enta fen dlwa2ty"  #@param {type:"string"}
show([t.strip() for t in text.split('|') if t.strip()])

## Export for the pip and npm packages
Turns the model above into the ONNX files the packages ship (`encoder.onnx`, `decoder.onnx`, `vocab.json`), checks them against the original on the test set, and saves them to your Drive in `MyDrive/egyptian-arabizi-franco/export/`. It keeps the small int8 version only if it scores exactly as well; otherwise it ships full precision.

Then download that `export` folder from Drive (or send it over) so it can go into the packages.

In [ ]:
!pip -q install onnx onnxruntime
EXPORT = '/content/drive/MyDrive/egyptian-arabizi-franco/export'
!python tools/export_onnx.py "$MODEL" --out "$EXPORT"
!ls -la "$EXPORT"